# GreenTensor — Full Capabilities Demo
**Carbon-aware, security-aware MLOps middleware for PyTorch**

Author: Dhivya Balakumar | v0.4.0 | MIT License

This notebook walks through every feature of GreenTensor:
1. Basic usage — wrap any training loop
2. Energy & carbon tracking
3. GPU optimizations
4. Baseline vs optimized comparison
5. Batch size optimization
6. Datacenter PUE impact
7. Carbon anomaly detection (power spikes, idle drain)
8. Digital footprint scanning — pre-deployment
9. Digital footprint scanning — post-deployment
10. Full security report
11. Dashboard walkthrough

## 0. Install

In [ ]:
# Install GreenTensor with full security stack
# !pip install greentensor[security]

# If running from source:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
print('GreenTensor ready')

---
## 1. Basic Usage — Wrap Any Training Loop
Drop `with GreenTensor()` around your existing code. Nothing else changes.

In [ ]:
import torch
import time
from greentensor import GreenTensor

def train():
    x = torch.randn(2000, 500)
    for _ in range(10):
        y = x @ x.T

print('Running training with GreenTensor...')
with GreenTensor(save_path='demo_metrics.pkl') as gt:
    with gt.mixed_precision():
        train()

print('\nMetrics captured:')
print(f'  Duration  : {gt.metrics.duration_s:.2f} s')
print(f'  Energy    : {gt.metrics.energy_kwh:.6f} kWh')
print(f'  CO2       : {gt.metrics.emissions_kg:.6f} kg')
print(f'  Idle time : {gt.metrics.idle_seconds:.2f} s')
print(f'  Saved to  : demo_metrics.pkl')

---
## 2. Baseline vs Optimized Comparison
Run the same workload twice — once without GreenTensor, once with — and compare real measured savings.

In [ ]:
import pickle
from greentensor.core.tracker import Tracker
from greentensor.report.metrics import RunMetrics, calculate_savings

# --- Baseline run (no optimizations) ---
tracker = Tracker()
tracker.start()
t0 = time.perf_counter()
train()
duration = time.perf_counter() - t0
emissions_kg, energy_kwh = tracker.stop()
baseline = RunMetrics(duration_s=duration, energy_kwh=energy_kwh, emissions_kg=emissions_kg)
print(f'Baseline  : {duration:.2f}s | {energy_kwh:.6f} kWh | {emissions_kg:.6f} kg CO2')

# --- Optimized run (with GreenTensor) ---
with GreenTensor(baseline=baseline, save_path='optimized_metrics.pkl', security=False) as gt:
    with gt.mixed_precision():
        train()

# --- Savings ---
savings = calculate_savings(baseline, gt.metrics)
print(f'\nSavings:')
print(f'  Energy saved    : {savings["energy_saved_kwh"]:.6f} kWh')
print(f'  CO2 saved       : {savings["emissions_saved_kg"]:.6f} kg')
print(f'  Reduction       : {savings["energy_reduction_pct"]:.1f}%')
print(f'  Time saved      : {savings["time_saved_s"]:.2f} s')

---
## 3. Batch Size Optimizer
GreenTensor queries available GPU memory and recommends the optimal batch size.

In [ ]:
from greentensor import optimize_batch_size, Config

config = Config(min_batch_size=32, max_batch_size=512)

for current in [16, 32, 64, 128]:
    recommended = optimize_batch_size(current, config)
    print(f'  Current batch: {current:4d}  ->  Recommended: {recommended}')

---
## 4. Datacenter PUE Impact
Your GPU reports X watts. The datacenter uses X * PUE watts total (cooling, UPS, networking).
GreenTensor scales your measurements to reflect real datacenter energy consumption.

In [ ]:
from greentensor import DatacenterConfig, PUE_PRESETS

# Simulate a baseline run result
sample = RunMetrics(duration_s=120.0, energy_kwh=0.0025, emissions_kg=0.00058)

print('PUE presets available:', list(PUE_PRESETS.keys()))
print()

scenarios = [
    ('Local workstation', DatacenterConfig(pue=1.0, carbon_intensity_kg_per_kwh=0.000233, num_nodes=1)),
    ('Cloud average (AWS/GCP)', DatacenterConfig(pue=1.2, carbon_intensity_kg_per_kwh=0.000386, num_nodes=1)),
    ('Enterprise DC', DatacenterConfig(pue=1.5, carbon_intensity_kg_per_kwh=0.000490, num_nodes=1)),
    ('8-node distributed (cloud)', DatacenterConfig(pue=1.2, carbon_intensity_kg_per_kwh=0.000386, num_nodes=8)),
]

print(f'{"Scenario":<35} {"DC Energy (kWh)":>18} {"DC CO2 (kg)":>14}')
print('-' * 70)
for name, dc in scenarios:
    adjusted = sample.apply_datacenter_config(dc)
    print(f'{name:<35} {adjusted.energy_kwh_dc:>18.6f} {adjusted.emissions_kg_dc:>14.6f}')

---
## 5. Carbon Anomaly Detection
GreenTensor monitors GPU power draw continuously.
Simulated here to show what alerts look like when a spike is detected.

In [ ]:
from greentensor.security.anomaly_detector import AnomalyDetector, AnomalyDetectorConfig, AnomalyAlert
from unittest.mock import patch

alerts_caught = []

config = AnomalyDetectorConfig(
    sample_interval_s=0.05,
    spike_threshold_pct=50.0,
    consecutive_anomalies_required=2,
    use_alibi_detect=False,
    use_llm_guard=False,
)
detector = AnomalyDetector(config=config, on_alert=alerts_caught.append)

call_count = [0]
def mock_gpu():
    call_count[0] += 1
    # Normal for first 6 samples, then spike
    return {'util_%': 80.0, 'power_W': 100.0} if call_count[0] <= 6 else {'util_%': 90.0, 'power_W': 280.0}

with patch('greentensor.core.profiler.Profiler.get_gpu_metrics', side_effect=mock_gpu):
    detector.start()
    time.sleep(0.6)
    detector.stop()

print(f'Alerts detected: {len(alerts_caught)}')
for a in alerts_caught:
    print(f'  [{a.severity.upper()}] [{a.alert_type}] [{a.source}]')
    print(f'  {a.message}')

---
## 6. Digital Footprint Scanner — Pre-Deployment
Detects model tampering, supply chain attacks, process injection, and network exfiltration
during the training phase.

In [ ]:
import tempfile, os
from greentensor.security.digital_footprint import DigitalFootprintScanner

# --- Model weight tampering detection ---
print('=== Model Weight Tampering Detection ===')
with tempfile.NamedTemporaryFile(delete=False, suffix='.pt') as f:
    f.write(b'original model weights v1.0')
    model_path = f.name

scanner = DigitalFootprintScanner(
    stage='pre_deployment',
    monitor_network=False,
    monitor_processes=False,
)
scanner.register_model_file(model_path)
print(f'Model registered: {os.path.basename(model_path)}')

# Simulate tampering
with open(model_path, 'wb') as f:
    f.write(b'BACKDOORED weights - malicious payload injected')
print('Model file tampered (simulated)...')

events = scanner.verify_model_files()
for e in events:
    print(f'\n  [{e.severity.upper()}] {e.category} | MITRE: {e.mitre_technique}')
    print(f'  {e.message}')
    print(f'  Evidence: original={e.evidence["original_hash"][:16]}...')
    print(f'            current ={e.evidence["current_hash"][:16]}...')

os.unlink(model_path)

In [ ]:
# --- Supply chain / dependency scan ---
print('=== Supply Chain Dependency Scan ===')
scanner2 = DigitalFootprintScanner(
    stage='pre_deployment',
    monitor_network=False,
    monitor_processes=False,
)
events = scanner2.scan_dependencies()
if events:
    for e in events:
        print(f'  [{e.severity.upper()}] {e.message}')
else:
    print('  No suspicious packages found — supply chain clean.')

---
## 7. Digital Footprint Scanner — Post-Deployment
Detects inference anomalies, model extraction attempts, and backdoor triggers
during production serving.

In [ ]:
print('=== Post-Deployment Inference Monitoring ===')

post_events = []
scanner3 = DigitalFootprintScanner(
    stage='post_deployment',
    on_event=post_events.append,
    monitor_network=False,
    monitor_processes=False,
)

# Simulate normal inference
print('Simulating 15 normal inference calls (10ms each)...')
for _ in range(15):
    scanner3.record_inference(latency_s=0.010, confidence=0.85)

# Simulate backdoor trigger (latency spike)
print('Simulating backdoor trigger (latency spike)...')
scanner3.record_inference(latency_s=0.150, confidence=0.9999)

# Simulate model extraction (rapid probing)
print('Simulating model extraction attack (rapid API calls)...')
t0 = time.time() - 0.025
for i in range(25):
    scanner3._api_call_times.append(t0 + i * 0.001)
    scanner3._inference_latencies.append(0.001)
scanner3.record_inference(latency_s=0.001)

print(f'\nPost-deployment events detected: {len(post_events)}')
for e in post_events:
    print(f'  [{e.severity.upper()}] [{e.category}] {e.signal}')
    print(f'  MITRE: {e.mitre_technique}')
    print(f'  {e.message}')

---
## 8. Full GreenTensor Session with Security
Everything together — optimizations, tracking, anomaly detection, digital footprint, auto-save.

In [ ]:
from greentensor.security.anomaly_detector import AnomalyDetectorConfig

security_config = AnomalyDetectorConfig(
    spike_threshold_pct=80.0,
    use_alibi_detect=False,  # set True if alibi-detect is installed
    use_llm_guard=False,     # set True if llm-guard is installed
)

def my_alert_handler(event):
    print(f'  ALERT: [{event.severity.upper()}] {event.message[:80]}...')

print('Starting full GreenTensor session...')
with GreenTensor(
    baseline=baseline,
    security=True,
    security_config=security_config,
    on_alert=my_alert_handler,
    save_path='full_session_metrics.pkl',
    stage='pre_deployment',
) as gt:
    with gt.mixed_precision():
        train()

print(f'\nSession complete.')
print(f'Security alerts : {len(gt.security_alerts)}')
print(f'Metrics saved to: full_session_metrics.pkl')

---
## 9. Load Saved Metrics and Visualize
Load the auto-saved .pkl files and compute savings — same as what the dashboard does.

In [ ]:
import pickle

with open('full_session_metrics.pkl', 'rb') as f:
    loaded = pickle.load(f)

print('Loaded from .pkl file:')
print(f'  duration_s   : {loaded.duration_s:.2f}')
print(f'  energy_kwh   : {loaded.energy_kwh:.6f}')
print(f'  emissions_kg : {loaded.emissions_kg:.6f}')
print(f'  idle_seconds : {loaded.idle_seconds:.2f}')

savings = calculate_savings(baseline, loaded)
print(f'\nSavings vs baseline:')
print(f'  Energy reduction : {savings["energy_reduction_pct"]:.1f}%')
print(f'  CO2 saved        : {savings["emissions_saved_kg"]:.6f} kg')
print(f'  Time saved       : {savings["time_saved_s"]:.2f} s')

---
## 10. Decorator Usage
Use `@GreenTensor.profile` to wrap a function without a context manager.

In [ ]:
@GreenTensor.profile
def my_training_job():
    x = torch.randn(1000, 500)
    for _ in range(5):
        y = x @ x.T
    return 'done'

result = my_training_job()
print('Decorator result:', result)

---
## 11. Custom Alert Handler + On-Alert Callback
Plug in your own handler — send Slack, write to a SIEM, kill the job.

In [ ]:
import json

audit_log = []

def audit_handler(event):
    """Write every security event to an audit log as JSON."""
    entry = {
        'timestamp': event.timestamp,
        'stage': getattr(event, 'stage', 'unknown'),
        'severity': event.severity,
        'type': event.alert_type if hasattr(event, 'alert_type') else event.category,
        'message': event.message,
    }
    audit_log.append(entry)

with GreenTensor(security=True, on_alert=audit_handler, verbose=False) as gt:
    train()

print(f'Audit log entries: {len(audit_log)}')
for entry in audit_log:
    print(json.dumps(entry, indent=2))

---
## 12. Summary — What GreenTensor Does

| Feature | How | Benefit |
|---------|-----|---------|
| Energy tracking | CodeCarbon + nvidia-smi fallback | Know exact kWh per training run |
| CO2 tracking | Grid carbon intensity x energy | Quantify environmental impact |
| GPU optimization | cuDNN benchmark + FP16 mixed precision | 20-40% energy reduction |
| Idle detection | Background thread, 0.5s sampling | Identify wasted GPU time |
| Batch optimization | GPU memory query | Maximize throughput |
| Datacenter PUE | PUE x nodes multiplier | Real DC energy, not just GPU |
| Carbon anomaly detection | alibi-detect SpectralResidual | Detect cryptominers, hidden processes |
| Model tampering | SHA-256 hash verification | Catch backdoor injection |
| Supply chain scan | Package name analysis | Catch typosquatting attacks |
| Process injection | psutil process monitoring | Detect unauthorized spawned processes |
| Network exfiltration | TCP connection monitoring | Detect data theft |
| Inference anomaly | Latency + confidence tracking | Detect backdoor triggers in production |
| Model extraction | API rate analysis | Detect model stealing attacks |
| Auto-save | pickle serialization | Load results into dashboard |
| Dashboard | Streamlit 5-page UI | Visualize everything |

```bash
# Launch the dashboard
streamlit run dashboard/app.py
```